In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 - Gold: Modelagem Analítica
# MAGIC
# MAGIC Objetivo:
# MAGIC - Ler as tabelas tratadas da camada Silver
# MAGIC - Construir dimensões e fatos para consumo analítico
# MAGIC - Criar marts prontos para BI
# MAGIC - Disponibilizar indicadores de negócio como receita, pedidos, ticket médio, cancelamento e atraso

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Volumes/case/default/case_data"

SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)

display(dbutils.fs.ls(SILVER_PATH))

SILVER_PATH: /Volumes/case/default/case_data/silver
GOLD_PATH: /Volumes/case/default/case_data/gold


path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/silver/canais/,canais/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/clientes/,clientes/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/entregas/,entregas/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/ocorrencias/,ocorrencias/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/pedidos_cabecalho/,pedidos_cabecalho/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/pedidos_itens/,pedidos_itens/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/produtos/,produtos/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/regioes/,regioes/,0,1781144055084
dbfs:/Volumes/case/default/case_data/silver/vendedores/,vendedores/,0,1781144055084


In [0]:
silver_pedidos_cabecalho = spark.read.format("delta").load(f"{SILVER_PATH}/pedidos_cabecalho")
silver_pedidos_itens = spark.read.format("delta").load(f"{SILVER_PATH}/pedidos_itens")
silver_produtos = spark.read.format("delta").load(f"{SILVER_PATH}/produtos")
silver_clientes = spark.read.format("delta").load(f"{SILVER_PATH}/clientes")
silver_canais = spark.read.format("delta").load(f"{SILVER_PATH}/canais")
silver_vendedores = spark.read.format("delta").load(f"{SILVER_PATH}/vendedores")
silver_regioes = spark.read.format("delta").load(f"{SILVER_PATH}/regioes")
silver_entregas = spark.read.format("delta").load(f"{SILVER_PATH}/entregas")
silver_ocorrencias = spark.read.format("delta").load(f"{SILVER_PATH}/ocorrencias")

print("Tabelas Silver carregadas com sucesso.")

Tabelas Silver carregadas com sucesso.


In [0]:
dim_cliente = (
    silver_clientes
    .select(
        "customer_id",
        "nome_cliente",
        "segmento",
        "porte",
        "cidade",
        "estado",
        "data_cadastro",
        "email",
        "status_cliente",
        "updated_at"
    )
    .dropDuplicates(["customer_id"])
)

dim_produto = (
    silver_produtos
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "product_status",
        "list_price",
        "currency",
        "product_family",
        "tags_text",
        "updated_at"
    )
    .dropDuplicates(["product_id"])
)

dim_canal = (
    silver_canais
    .select(
        "channel_id",
        "nome_canal",
        "tipo_canal",
        "status_canal",
        "observacao"
    )
    .dropDuplicates(["channel_id"])
)

dim_vendedor = (
    silver_vendedores
    .select(
        "seller_id",
        "seller_name",
        "channel_id",
        "regional_code",
        "hire_date",
        "status_vendedor"
    )
    .dropDuplicates(["seller_id"])
)

dim_regiao = (
    silver_regioes
    .select(
        "regional_code",
        "regional_name",
        "state",
        "manager_name",
        "status_regiao"
    )
    .dropDuplicates(["regional_code", "state"])
)

print("Dimensões criadas.")
display(dim_cliente.limit(5))
display(dim_produto.limit(5))
display(dim_canal.limit(5))
display(dim_vendedor.limit(5))
display(dim_regiao.limit(5))

Dimensões criadas.


customer_id,nome_cliente,segmento,porte,cidade,estado,data_cadastro,email,status_cliente,updated_at
C0008,Cliente 8,null,MÉDIA,Joinville,SC,2024-02-16T00:00:00.000Z,cliente8@empresa.com,ATIVO,2025-07-14T00:00:00.000Z
C0015,Cliente 15,Serviços,MÉDIA,Curitiba,PR,2025-02-12T00:00:00.000Z,cliente15@empresa.com,ATIVO,2026-01-22T00:00:00.000Z
C0038,Cliente 38,Varejo,MÉDIA,Florianópolis,SC,2025-06-04T00:00:00.000Z,cliente38@empresa.com,NAN,2026-03-18T00:00:00.000Z
C0045,Cliente 45,null,PEQUENA,Curitiba,PR,2024-11-02T00:00:00.000Z,cliente45@empresa.com,ATIVO,2026-01-16T00:00:00.000Z
C0050,Cliente 50,Varejo,GRANDE,Ribeirão Preto,SP,2025-07-12T00:00:00.000Z,cliente50@empresa.com,ATIVO,2025-01-22T00:00:00.000Z


product_id,product_name,category,subcategory,product_status,list_price,currency,product_family,tags_text,updated_at
P0002,Produto 2,Hardware,Sensor,ATIVO,2934.56,BRL,Premium,"b2b,cloud",2025-03-08T00:00:00.000Z
P0004,Produto 4,Assinatura,Anual,ATIVO,1844.86,BRL,Premium,b2b,2026-01-28T00:00:00.000Z
P0007,Produto 7,Assinatura,Mensal,INATIVO,945.19,BRL,Premium,"novo,cloud,promo",2025-12-13T00:00:00.000Z
P0014,Produto 14,Software,CRM,ATIVO,3967.33,BRL,Premium,,2025-05-27T00:00:00.000Z
P0023,Produto 23,Assinatura,Mensal,ATIVO,1047.88,BRL,Premium,,2025-06-11T00:00:00.000Z


channel_id,nome_canal,tipo_canal,status_canal,observacao
CH07,Revendas,INDIRETO,NAN,id em lowercase
CH01,Inside Sales,Direto,ATIVO,null
CH06,null,Indireto,ATIVO,nome ausente
CH05,E-commerce,digital,ATIVO,null
CH02,Parceiros,Indireto,ATIVO,null


seller_id,seller_name,channel_id,regional_code,hire_date,status_vendedor
V033,Vendedor 33,CH01,SUL,2025-06-03T00:00:00.000Z,null
V002,Vendedor 2,CH04,S,2023-12-16T00:00:00.000Z,ATIVO
V029,Vendedor 29,CH02,S,2023-03-11T00:00:00.000Z,null
V032,Vendedor 32,CH04,N,2023-02-28T00:00:00.000Z,ATIVO
V034,Vendedor 34,CH04,CO,2024-07-21T00:00:00.000Z,null


regional_code,regional_name,state,manager_name,status_regiao
SE,Sudeste,SP,Ana Costa,ATIVO
XX,null,null,Sem gestor,INATIVO
N,Norte,AM,Mariana Lopes,ATIVO
SUL,Região Sul,SC,Rafael Souza,ATIVO
CO,Centro-Oeste,GO,Paulo Teixeira,ATIVO


In [0]:
datas_pedidos = silver_pedidos_cabecalho.select(F.to_date("order_date").alias("data"))
datas_promessa = silver_pedidos_cabecalho.select(F.to_date("promised_date").alias("data"))
datas_entrega_envio = silver_entregas.select(F.to_date("shipped_at").alias("data"))
datas_entrega_realizada = silver_entregas.select(F.to_date("delivered_at").alias("data"))
datas_ocorrencias = silver_ocorrencias.select(F.to_date("created_at").alias("data"))

dim_data = (
    datas_pedidos
    .union(datas_promessa)
    .union(datas_entrega_envio)
    .union(datas_entrega_realizada)
    .union(datas_ocorrencias)
    .where(F.col("data").isNotNull())
    .dropDuplicates(["data"])
    .withColumn("data_id", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("dia_semana", F.date_format("data", "E"))
)

display(dim_data.orderBy("data").limit(10))

data,data_id,ano,mes,dia,trimestre,ano_mes,dia_semana
2025-01-01,20250101,2025,1,1,1,2025-01,Wed
2025-01-03,20250103,2025,1,3,1,2025-01,Fri
2025-01-04,20250104,2025,1,4,1,2025-01,Sat
2025-01-05,20250105,2025,1,5,1,2025-01,Sun
2025-01-06,20250106,2025,1,6,1,2025-01,Mon
2025-01-07,20250107,2025,1,7,1,2025-01,Tue
2025-01-08,20250108,2025,1,8,1,2025-01,Wed
2025-01-09,20250109,2025,1,9,1,2025-01,Thu
2025-01-10,20250110,2025,1,10,1,2025-01,Fri
2025-01-11,20250111,2025,1,11,1,2025-01,Sat


In [0]:
fato_pedido = (
    silver_pedidos_cabecalho
    .select(
        "order_id",
        "customer_id",
        "seller_id",
        F.to_date("order_date").alias("order_date"),
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("order_date_id"),
        F.to_date("promised_date").alias("promised_date"),
        F.date_format("promised_date", "yyyyMMdd").cast("int").alias("promised_date_id"),
        "status_order",
        "gross_amount",
        "discount_amount",
        "net_amount",
        "payment_priority",
        "payment_source",
        "dias_ate_promessa",
        "fl_cancelado"
    )
)

display(fato_pedido.limit(5))

order_id,customer_id,seller_id,order_date,order_date_id,promised_date,promised_date_id,status_order,gross_amount,discount_amount,net_amount,payment_priority,payment_source,dias_ate_promessa,fl_cancelado
O00001,C0002,V036,2025-07-31,20250731,2025-08-06,20250806,FATURADO,10788.64,0.0,10788.64,null,null,6,0
O00002,C0023,V008,2025-04-13,20250413,2025-04-26,20250426,EM_SEPARACAO,14462.32,0.0,14462.32,null,null,13,0
O00003,C0127,V012,2025-04-01,20250401,2025-04-16,20250416,FATURADO,13802.56,1994.35,11808.21,null,null,15,0
O00004,C0165,V035,2025-11-05,20251105,2025-11-07,20251107,CANCELADO,13933.98,670.75,13263.23,null,null,2,1
O00005,C0048,V001,2026-02-19,20260219,2026-02-21,20260221,null,5027.19,0.0,5027.19,null,null,2,0


In [0]:
fato_pedido_item = (
    silver_pedidos_itens
    .select(
        "order_id",
        "item_seq",
        "product_id",
        "quantity",
        "unit_price",
        "total_item",
        "valor_calculado_item",
        "dif_total_item",
        "item_status"
    )
)

display(fato_pedido_item.limit(5))

order_id,item_seq,product_id,quantity,unit_price,total_item,valor_calculado_item,dif_total_item,item_status
O00001,1,P0027,5.0,453.12,2265.6,2265.6,0.0,ATIVO
O00001,2,P0035,10.0,2278.1,22781.0,22781.0,0.0,null
O00001,3,P0063,2.0,629.8,1259.6,1259.6,0.0,ATIVO
O00001,4,P0001,3.0,1502.54,4507.62,4507.62,0.0,ATIVO
O00002,1,P0032,3.0,1274.78,3824.34,3824.34,0.0,ATIVO


In [0]:
fato_entrega = (
    silver_entregas
    .select(
        "delivery_id",
        "order_id",
        "carrier_name",
        "delivery_mode",
        "delivery_status",
        F.to_date("shipped_at").alias("shipped_at"),
        F.date_format("shipped_at", "yyyyMMdd").cast("int").alias("shipped_date_id"),
        F.to_date("delivered_at").alias("delivered_at"),
        F.date_format("delivered_at", "yyyyMMdd").cast("int").alias("delivered_date_id"),
        "destination_state",
        "destination_city",
        "delivery_cost",
        "delivery_days",
        "fl_entregue",
        "fl_entrega_atrasada"
    )
)

display(fato_entrega.limit(5))

delivery_id,order_id,carrier_name,delivery_mode,delivery_status,shipped_at,shipped_date_id,delivered_at,delivered_date_id,destination_state,destination_city,delivery_cost,delivery_days,fl_entregue,fl_entrega_atrasada
D00001,O00129,null,Rodoviário,null,2026-01-21,20260121,2026-01-21,20260121,RJ,Rio de Janeiro,56.54,0,1,0
D00002,O00213,TransSul,null,ENTREGUE,2025-02-18,20250218,2025-02-27,20250227,SC,Blumenau,46.34,9,1,0
D00003,O00048,LogFast,Rodoviário,ATRASADO,2025-03-23,20250323,2025-04-01,20250401,PR,Maringá,248.24,9,1,1
D00004,O00250,null,Aéreo,null,2025-02-23,20250223,2025-03-05,20250305,SC,Florianópolis,346.48,10,1,0
D00005,O00172,EntregaJá,Aéreo,ENTREGUE,2025-01-03,20250103,2025-01-13,20250113,PR,Curitiba,218.51,10,1,0


In [0]:
fato_ocorrencia = (
    silver_ocorrencias
    .select(
        "ticket_id",
        "order_id",
        "event_type",
        F.to_date("created_at").alias("created_at"),
        F.date_format("created_at", "yyyyMMdd").cast("int").alias("created_date_id"),
        "severity",
        "status_ocorrencia",
        "fl_ocorrencia_aberta"
    )
)

display(fato_ocorrencia.limit(5))

ticket_id,order_id,event_type,created_at,created_date_id,severity,status_ocorrencia,fl_ocorrencia_aberta
T00001,O00385,refund,2025-01-04,20250104,HIGH,FECHADO,0
T00002,O00051,troca,2026-02-12,20260212,MEDIUM,ABERTO,1
T00003,O00181,refund,2026-02-07,20260207,null,null,0
T00004,O00371,delay,2025-12-18,20251218,LOW,null,0
T00005,O00055,null,2025-02-27,20250227,MEDIUM,ABERTO,1


In [0]:
mart_performance_comercial = (
    fato_pedido.alias("p")
    .join(
        dim_cliente.alias("c"),
        F.col("p.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .join(
        dim_vendedor.alias("v"),
        F.col("p.seller_id") == F.col("v.seller_id"),
        "left"
    )
    .join(
        dim_canal.alias("can"),
        F.col("v.channel_id") == F.col("can.channel_id"),
        "left"
    )
    .join(
        dim_regiao.alias("r"),
        F.col("v.regional_code") == F.col("r.regional_code"),
        "left"
    )
    .groupBy(
        F.col("p.order_date").alias("data_pedido"),
        F.date_format(F.col("p.order_date"), "yyyy-MM").alias("ano_mes"),
        F.col("c.estado").alias("estado_cliente"),
        F.col("c.segmento").alias("segmento_cliente"),
        F.col("c.porte").alias("porte_cliente"),
        F.col("v.seller_id"),
        F.col("v.seller_name"),
        F.col("can.channel_id"),
        F.col("can.nome_canal"),
        F.col("can.tipo_canal"),
        F.col("r.regional_code"),
        F.col("r.regional_name")
    )
    .agg(
        F.countDistinct("p.order_id").alias("qtd_pedidos"),
        F.round(F.sum("p.gross_amount"), 2).alias("receita_bruta"),
        F.round(F.sum("p.discount_amount"), 2).alias("valor_desconto"),
        F.round(F.sum("p.net_amount"), 2).alias("receita_liquida"),
        F.sum("p.fl_cancelado").alias("qtd_pedidos_cancelados"),
        F.round(F.avg("p.net_amount"), 2).alias("ticket_medio")
    )
    .withColumn(
        "taxa_cancelamento",
        F.round(
            F.when(F.col("qtd_pedidos") > 0, F.col("qtd_pedidos_cancelados") / F.col("qtd_pedidos"))
             .otherwise(F.lit(0)),
            4
        )
    )
)

display(mart_performance_comercial.limit(10))

data_pedido,ano_mes,estado_cliente,segmento_cliente,porte_cliente,seller_id,seller_name,channel_id,nome_canal,tipo_canal,regional_code,regional_name,qtd_pedidos,receita_bruta,valor_desconto,receita_liquida,qtd_pedidos_cancelados,ticket_medio,taxa_cancelamento
2025-12-21,2025-12,RJ,Educação,null,V018,Vendedor 18,CH02,Parceiros,Indireto,null,null,1,8545.08,0.0,8545.08,1,8545.08,1.0
2025-04-22,2025-04,SP,Serviços,null,V022,Vendedor 22,CH03,Marketplace,Digital,null,null,1,12390.23,0.0,12390.23,0,12390.23,0.0
2025-05-19,2025-05,PR,Educação,GRANDE,V009,Vendedor 9,CH01,Inside Sales,Direto,NE,Nordeste,1,5104.78,0.0,5104.78,0,5104.78,0.0
2025-08-13,2025-08,MG,Varejo,null,V012,Vendedor 12,CH01,Inside Sales,Direto,S,Sul,1,4097.81,0.0,4097.81,0,4097.81,0.0
2025-12-18,2025-12,MG,Indústria,GRANDE,V008,Vendedor 8,CH01,Inside Sales,Direto,SE,Sudeste,1,12913.44,0.0,12913.44,0,12913.44,0.0
2025-06-01,2025-06,SP,Educação,MÉDIA,V025,Vendedor 25,CH01,Inside Sales,Direto,NE,Nordeste,1,913.08,0.0,913.08,0,913.08,0.0
2025-08-12,2025-08,SC,Serviços,null,V030,Vendedor 30,CH07,Revendas,INDIRETO,SE,Sudeste,1,12617.64,0.0,12617.64,0,12617.64,0.0
2025-05-07,2025-05,MG,Varejo,GRANDE,V011,Vendedor 11,null,null,null,SE,Sudeste,1,4334.21,0.0,4334.21,0,4334.21,0.0
2025-06-16,2025-06,PR,Educação,GRANDE,V025,Vendedor 25,CH01,Inside Sales,Direto,NE,Nordeste,1,7705.62,1549.71,6155.91,0,6155.91,0.0
2025-10-31,2025-10,RJ,null,PEQUENA,V023,Vendedor 23,CH05,E-commerce,digital,CO,Centro-Oeste,1,12112.53,0.0,12112.53,0,12112.53,0.0


In [0]:
mart_performance_produtos = (
    fato_pedido_item.alias("i")
    .join(
        fato_pedido.alias("p"),
        F.col("i.order_id") == F.col("p.order_id"),
        "left"
    )
    .join(
        dim_produto.alias("prod"),
        F.col("i.product_id") == F.col("prod.product_id"),
        "left"
    )
    .groupBy(
        F.col("p.order_date").alias("data_pedido"),
        F.date_format(F.col("p.order_date"), "yyyy-MM").alias("ano_mes"),
        F.col("prod.product_id"),
        F.col("prod.product_name"),
        F.col("prod.category"),
        F.col("prod.subcategory"),
        F.col("prod.product_family")
    )
    .agg(
        F.countDistinct("i.order_id").alias("qtd_pedidos"),
        F.round(F.sum("i.quantity"), 2).alias("qtd_itens_vendidos"),
        F.round(F.sum("i.total_item"), 2).alias("receita_item"),
        F.round(F.avg("i.unit_price"), 2).alias("preco_medio_unitario")
    )
)

display(mart_performance_produtos.limit(10))

data_pedido,ano_mes,product_id,product_name,category,subcategory,product_family,qtd_pedidos,qtd_itens_vendidos,receita_item,preco_medio_unitario
2025-04-13,2025-04,P0056,Produto 56,Software,ETL,Legacy,1,2.0,3625.44,1812.72
2026-01-21,2026-01,P0044,Produto 44,Software,null,Premium,1,3.0,7314.96,2438.32
2025-01-06,2025-01,P0036,Produto 36,Serviços,Suporte,Legacy,1,0.0,0.0,2143.35
2025-05-28,2025-05,P0047,Produto 47,Software,CRM,null,1,1.0,2992.91,2992.91
2025-04-19,2025-04,P0002,Produto 2,Hardware,Sensor,Premium,1,1.0,290.37,290.37
2025-08-31,2025-08,P0068,Produto 68,Serviços,Implantação,Core,1,5.0,11779.6,2355.92
2026-02-02,2026-02,P0037,Produto 37,Serviços,Implantação,Premium,1,10.0,9705.3,970.53
2026-01-21,2026-01,P0009,Produto 9,Serviços,Suporte,null,1,5.0,14417.7,2883.54
2026-02-02,2026-02,P0039,Produto 39,Serviços,Suporte,null,1,5.0,8977.55,1795.51
2025-12-17,2025-12,P0058,Produto 58,Hardware,Sensor,null,1,-1.0,-2242.07,2242.07


In [0]:
mart_operacional_entregas = (
    fato_entrega.alias("e")
    .join(
        fato_pedido.alias("p"),
        F.col("e.order_id") == F.col("p.order_id"),
        "left"
    )
    .join(
        dim_cliente.alias("c"),
        F.col("p.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .groupBy(
        F.col("e.shipped_at").alias("data_envio"),
        F.date_format(F.col("e.shipped_at"), "yyyy-MM").alias("ano_mes"),
        F.col("e.destination_state"),
        F.col("e.destination_city"),
        F.col("e.carrier_name"),
        F.col("e.delivery_mode"),
        F.col("e.delivery_status"),
        F.col("c.segmento").alias("segmento_cliente"),
        F.col("c.porte").alias("porte_cliente")
    )
    .agg(
        F.countDistinct("e.delivery_id").alias("qtd_entregas"),
        F.countDistinct("e.order_id").alias("qtd_pedidos_com_entrega"),
        F.sum("e.fl_entregue").alias("qtd_entregues"),
        F.sum("e.fl_entrega_atrasada").alias("qtd_entregas_atrasadas"),
        F.round(F.avg("e.delivery_days"), 2).alias("prazo_medio_entrega_dias"),
        F.round(F.sum("e.delivery_cost"), 2).alias("custo_total_entrega"),
        F.round(F.avg("e.delivery_cost"), 2).alias("custo_medio_entrega")
    )
    .withColumn(
        "taxa_atraso_entrega",
        F.round(
            F.when(F.col("qtd_entregas") > 0, F.col("qtd_entregas_atrasadas") / F.col("qtd_entregas"))
             .otherwise(F.lit(0)),
            4
        )
    )
)

display(mart_operacional_entregas.limit(10))

data_envio,ano_mes,destination_state,destination_city,carrier_name,delivery_mode,delivery_status,segmento_cliente,porte_cliente,qtd_entregas,qtd_pedidos_com_entrega,qtd_entregues,qtd_entregas_atrasadas,prazo_medio_entrega_dias,custo_total_entrega,custo_medio_entrega,taxa_atraso_entrega
2025-08-31,2025-08,SC,Florianópolis,LogFast,rodoviario,ENTREGUE,Indústria,MÉDIA,1,1,1,0,6.0,269.9,269.9,0.0
2026-02-09,2026-02,SC,Joinville,LogFast,null,ENTREGUE,null,MÉDIA,1,1,1,0,3.0,65.98,65.98,0.0
2025-03-22,2025-03,PR,Londrina,LogFast,Rodoviário,EM_TRANSITO,null,PEQUENA,1,1,1,0,3.0,152.21,152.21,0.0
2025-10-25,2025-10,SC,Joinville,EntregaJá,rodoviario,CANCELADO,Educação,MÉDIA,1,1,1,0,11.0,208.19,208.19,0.0
2025-03-09,2025-03,RJ,Volta Redonda,LogFast,Aéreo,ATRASADO,Saúde,MÉDIA,1,1,1,1,5.0,325.24,325.24,1.0
2025-08-26,2025-08,SP,São Paulo,LogFast,Rodoviário,ENTREGUE,Educação,GRANDE,1,1,1,0,0.0,84.42,84.42,0.0
2025-09-08,2025-09,MG,Belo Horizonte,null,Aéreo,EM_TRANSITO,Educação,null,1,1,1,0,11.0,204.15,204.15,0.0
2025-05-01,2025-05,SC,Florianópolis,EntregaJá,Rodoviário,ENTREGUE,Educação,PEQUENA,1,1,1,0,6.0,306.48,306.48,0.0
2025-01-17,2025-01,SC,Joinville,TransSul,Rodoviário,CANCELADO,Indústria,MÉDIA,1,1,0,0,null,201.91,201.91,0.0
2025-01-14,2025-01,MG,Uberlândia,LogFast,Rodoviário,ENTREGUE,Serviços,null,1,1,1,0,12.0,65.04,65.04,0.0


In [0]:
mart_atendimento_ocorrencias = (
    fato_ocorrencia.alias("o")
    .join(
        fato_pedido.alias("p"),
        F.col("o.order_id") == F.col("p.order_id"),
        "left"
    )
    .join(
        dim_cliente.alias("c"),
        F.col("p.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .groupBy(
        F.col("o.created_at").alias("data_ocorrencia"),
        F.date_format(F.col("o.created_at"), "yyyy-MM").alias("ano_mes"),
        F.col("o.event_type"),
        F.col("o.severity"),
        F.col("o.status_ocorrencia"),
        F.col("c.estado").alias("estado_cliente"),
        F.col("c.segmento").alias("segmento_cliente"),
        F.col("c.porte").alias("porte_cliente")
    )
    .agg(
        F.countDistinct("o.ticket_id").alias("qtd_ocorrencias"),
        F.countDistinct("o.order_id").alias("qtd_pedidos_com_ocorrencia"),
        F.sum("o.fl_ocorrencia_aberta").alias("qtd_ocorrencias_abertas")
    )
    .withColumn(
        "taxa_ocorrencia_aberta",
        F.round(
            F.when(F.col("qtd_ocorrencias") > 0, F.col("qtd_ocorrencias_abertas") / F.col("qtd_ocorrencias"))
             .otherwise(F.lit(0)),
            4
        )
    )
)

display(mart_atendimento_ocorrencias.limit(10))

data_ocorrencia,ano_mes,event_type,severity,status_ocorrencia,estado_cliente,segmento_cliente,porte_cliente,qtd_ocorrencias,qtd_pedidos_com_ocorrencia,qtd_ocorrencias_abertas,taxa_ocorrencia_aberta
2026-02-07,2026-02,refund,null,null,SP,null,GRANDE,1,1,0,0.0
2026-01-03,2026-01,cancel_request,MEDIUM,ABERTO,SC,Serviços,GRANDE,1,1,1,1.0
2025-08-24,2025-08,complaint,null,ABERTO,SP,Educação,GRANDE,1,1,1,1.0
2025-12-03,2025-12,delay,HIGH,ABERTO,MG,Saúde,null,1,1,1,1.0
2025-10-14,2025-10,Delay,MEDIUM,ABERTO,PR,null,PEQUENA,1,1,1,1.0
2025-11-27,2025-11,refund,LOW,null,SC,Educação,MÉDIA,1,1,0,0.0
2025-05-15,2025-05,null,MEDIUM,FECHADO,RJ,Varejo,GRANDE,1,1,0,0.0
2026-01-22,2026-01,delay,HIGH,FECHADO,RJ,null,GRANDE,1,1,0,0.0
2025-02-26,2025-02,cancel_request,MEDIUM,null,SP,Saúde,PEQUENA,1,1,0,0.0
2025-04-04,2025-04,cancel_request,MEDIUM,ABERTO,null,null,null,1,1,1,1.0


In [0]:
tabelas_gold = {
    "dim_cliente": dim_cliente,
    "dim_produto": dim_produto,
    "dim_canal": dim_canal,
    "dim_vendedor": dim_vendedor,
    "dim_regiao": dim_regiao,
    "dim_data": dim_data,
    "fato_pedido": fato_pedido,
    "fato_pedido_item": fato_pedido_item,
    "fato_entrega": fato_entrega,
    "fato_ocorrencia": fato_ocorrencia,
    "mart_performance_comercial": mart_performance_comercial,
    "mart_performance_produtos": mart_performance_produtos,
    "mart_operacional_entregas": mart_operacional_entregas,
    "mart_atendimento_ocorrencias": mart_atendimento_ocorrencias
}

for nome_tabela, df in tabelas_gold.items():
    caminho_delta = f"{GOLD_PATH}/{nome_tabela}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_delta)
    )
    
    print(
        f"Tabela Gold gravada: {nome_tabela} | "
        f"Linhas: {df.count()} | "
        f"Caminho: {caminho_delta}"
    )

Tabela Gold gravada: dim_cliente | Linhas: 180 | Caminho: /Volumes/case/default/case_data/gold/dim_cliente
Tabela Gold gravada: dim_produto | Linhas: 71 | Caminho: /Volumes/case/default/case_data/gold/dim_produto
Tabela Gold gravada: dim_canal | Linhas: 7 | Caminho: /Volumes/case/default/case_data/gold/dim_canal
Tabela Gold gravada: dim_vendedor | Linhas: 40 | Caminho: /Volumes/case/default/case_data/gold/dim_vendedor
Tabela Gold gravada: dim_regiao | Linhas: 7 | Caminho: /Volumes/case/default/case_data/gold/dim_regiao
Tabela Gold gravada: dim_data | Linhas: 418 | Caminho: /Volumes/case/default/case_data/gold/dim_data
Tabela Gold gravada: fato_pedido | Linhas: 403 | Caminho: /Volumes/case/default/case_data/gold/fato_pedido
Tabela Gold gravada: fato_pedido_item | Linhas: 995 | Caminho: /Volumes/case/default/case_data/gold/fato_pedido_item
Tabela Gold gravada: fato_entrega | Linhas: 322 | Caminho: /Volumes/case/default/case_data/gold/fato_entrega
Tabela Gold gravada: fato_ocorrencia | Li

In [0]:
display(dbutils.fs.ls(GOLD_PATH))

for nome_tabela in tabelas_gold.keys():
    df_validacao = spark.read.format("delta").load(f"{GOLD_PATH}/{nome_tabela}")
    
    print("=" * 100)
    print(f"gold.{nome_tabela}")
    print("Linhas:", df_validacao.count())
    df_validacao.printSchema()
    display(df_validacao.limit(5))

path,name,size,modificationTime
dbfs:/Volumes/case/default/case_data/gold/dim_canal/,dim_canal/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/dim_cliente/,dim_cliente/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/dim_data/,dim_data/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/dim_produto/,dim_produto/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/dim_regiao/,dim_regiao/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/dim_vendedor/,dim_vendedor/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/fato_entrega/,fato_entrega/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/fato_ocorrencia/,fato_ocorrencia/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/fato_pedido/,fato_pedido/,0,1781144560906
dbfs:/Volumes/case/default/case_data/gold/fato_pedido_item/,fato_pedido_item/,0,1781144560906


gold.dim_cliente
Linhas: 180
root
 |-- customer_id: string (nullable = true)
 |-- nome_cliente: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- porte: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- status_cliente: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



customer_id,nome_cliente,segmento,porte,cidade,estado,data_cadastro,email,status_cliente,updated_at
C0008,Cliente 8,null,MÉDIA,Joinville,SC,2024-02-16T00:00:00.000Z,cliente8@empresa.com,ATIVO,2025-07-14T00:00:00.000Z
C0015,Cliente 15,Serviços,MÉDIA,Curitiba,PR,2025-02-12T00:00:00.000Z,cliente15@empresa.com,ATIVO,2026-01-22T00:00:00.000Z
C0038,Cliente 38,Varejo,MÉDIA,Florianópolis,SC,2025-06-04T00:00:00.000Z,cliente38@empresa.com,NAN,2026-03-18T00:00:00.000Z
C0045,Cliente 45,null,PEQUENA,Curitiba,PR,2024-11-02T00:00:00.000Z,cliente45@empresa.com,ATIVO,2026-01-16T00:00:00.000Z
C0050,Cliente 50,Varejo,GRANDE,Ribeirão Preto,SP,2025-07-12T00:00:00.000Z,cliente50@empresa.com,ATIVO,2025-01-22T00:00:00.000Z


gold.dim_produto
Linhas: 71
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- product_status: string (nullable = true)
 |-- list_price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- tags_text: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



product_id,product_name,category,subcategory,product_status,list_price,currency,product_family,tags_text,updated_at
P0002,Produto 2,Hardware,Sensor,ATIVO,2934.56,BRL,Premium,"b2b,cloud",2025-03-08T00:00:00.000Z
P0004,Produto 4,Assinatura,Anual,ATIVO,1844.86,BRL,Premium,b2b,2026-01-28T00:00:00.000Z
P0007,Produto 7,Assinatura,Mensal,INATIVO,945.19,BRL,Premium,"novo,cloud,promo",2025-12-13T00:00:00.000Z
P0014,Produto 14,Software,CRM,ATIVO,3967.33,BRL,Premium,,2025-05-27T00:00:00.000Z
P0023,Produto 23,Assinatura,Mensal,ATIVO,1047.88,BRL,Premium,,2025-06-11T00:00:00.000Z


...

data_pedido,ano_mes,product_id,product_name,category,subcategory,product_family,qtd_pedidos,qtd_itens_vendidos,receita_item,preco_medio_unitario
2025-04-13,2025-04,P0056,Produto 56,Software,ETL,Legacy,1,2.0,3625.44,1812.72
2026-01-21,2026-01,P0044,Produto 44,Software,null,Premium,1,3.0,7314.96,2438.32
2025-01-06,2025-01,P0036,Produto 36,Serviços,Suporte,Legacy,1,0.0,0.0,2143.35
2025-05-28,2025-05,P0047,Produto 47,Software,CRM,null,1,1.0,2992.91,2992.91
2025-04-19,2025-04,P0002,Produto 2,Hardware,Sensor,Premium,1,1.0,290.37,290.37


gold.mart_operacional_entregas
Linhas: 322
root
 |-- data_envio: date (nullable = true)
 |-- ano_mes: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- carrier_name: string (nullable = true)
 |-- delivery_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- segmento_cliente: string (nullable = true)
 |-- porte_cliente: string (nullable = true)
 |-- qtd_entregas: long (nullable = true)
 |-- qtd_pedidos_com_entrega: long (nullable = true)
 |-- qtd_entregues: long (nullable = true)
 |-- qtd_entregas_atrasadas: long (nullable = true)
 |-- prazo_medio_entrega_dias: double (nullable = true)
 |-- custo_total_entrega: double (nullable = true)
 |-- custo_medio_entrega: double (nullable = true)
 |-- taxa_atraso_entrega: double (nullable = true)



data_envio,ano_mes,destination_state,destination_city,carrier_name,delivery_mode,delivery_status,segmento_cliente,porte_cliente,qtd_entregas,qtd_pedidos_com_entrega,qtd_entregues,qtd_entregas_atrasadas,prazo_medio_entrega_dias,custo_total_entrega,custo_medio_entrega,taxa_atraso_entrega
2025-03-22,2025-03,PR,Londrina,LogFast,Rodoviário,EM_TRANSITO,null,PEQUENA,1,1,1,0,3.0,152.21,152.21,0.0
2025-12-25,2025-12,MG,Belo Horizonte,null,Aéreo,ENTREGUE,Educação,GRANDE,1,1,1,0,0.0,137.88,137.88,0.0
2025-05-09,2025-05,MG,Belo Horizonte,TransSul,Rodoviário,ENTREGUE,null,MÉDIA,1,1,1,0,7.0,23.17,23.17,0.0
2026-01-09,2026-01,MG,Uberlândia,null,Aéreo,ENTREGUE,Serviços,null,1,1,1,0,0.0,150.66,150.66,0.0
2025-04-28,2025-04,RJ,Rio de Janeiro,EntregaJá,Aéreo,ATRASADO,Varejo,GRANDE,1,1,1,1,11.0,169.74,169.74,1.0


gold.mart_atendimento_ocorrencias
Linhas: 270
root
 |-- data_ocorrencia: date (nullable = true)
 |-- ano_mes: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- status_ocorrencia: string (nullable = true)
 |-- estado_cliente: string (nullable = true)
 |-- segmento_cliente: string (nullable = true)
 |-- porte_cliente: string (nullable = true)
 |-- qtd_ocorrencias: long (nullable = true)
 |-- qtd_pedidos_com_ocorrencia: long (nullable = true)
 |-- qtd_ocorrencias_abertas: long (nullable = true)
 |-- taxa_ocorrencia_aberta: double (nullable = true)



data_ocorrencia,ano_mes,event_type,severity,status_ocorrencia,estado_cliente,segmento_cliente,porte_cliente,qtd_ocorrencias,qtd_pedidos_com_ocorrencia,qtd_ocorrencias_abertas,taxa_ocorrencia_aberta
2025-04-19,2025-04,cancel_request,LOW,null,MG,Varejo,GRANDE,1,1,0,0.0
2026-01-12,2026-01,troca,null,FECHADO,SC,Indústria,GRANDE,1,1,0,0.0
2026-01-28,2026-01,complaint,HIGH,ABERTO,RJ,Indústria,PEQUENA,1,1,1,1.0
2025-01-19,2025-01,troca,HIGH,FECHADO,RJ,Educação,GRANDE,1,1,0,0.0
2025-05-05,2025-05,complaint,null,ABERTO,SP,Varejo,null,1,1,1,1.0


In [0]:
print("Resumo de indicadores principais")

total_pedidos = fato_pedido.select("order_id").distinct().count()
total_clientes = dim_cliente.select("customer_id").distinct().count()
total_produtos = dim_produto.select("product_id").distinct().count()
total_itens = fato_pedido_item.count()
total_entregas = fato_entrega.select("delivery_id").distinct().count()
total_ocorrencias = fato_ocorrencia.select("ticket_id").distinct().count()

receita_liquida = (
    fato_pedido
    .agg(F.round(F.sum("net_amount"), 2).alias("receita_liquida"))
    .collect()[0]["receita_liquida"]
)

pedidos_cancelados = (
    fato_pedido
    .agg(F.sum("fl_cancelado").alias("cancelados"))
    .collect()[0]["cancelados"]
)

entregas_atrasadas = (
    fato_entrega
    .agg(F.sum("fl_entrega_atrasada").alias("atrasadas"))
    .collect()[0]["atrasadas"]
)

print(f"Total de pedidos: {total_pedidos}")
print(f"Total de clientes: {total_clientes}")
print(f"Total de produtos: {total_produtos}")
print(f"Total de itens de pedidos: {total_itens}")
print(f"Total de entregas: {total_entregas}")
print(f"Total de ocorrências: {total_ocorrencias}")
print(f"Receita líquida total: {receita_liquida}")
print(f"Pedidos cancelados: {pedidos_cancelados}")
print(f"Entregas atrasadas: {entregas_atrasadas}")

Resumo de indicadores principais
Total de pedidos: 400
Total de clientes: 180
Total de produtos: 71
Total de itens de pedidos: 995
Total de entregas: 320
Total de ocorrências: 270
Receita líquida total: 2909881.99
Pedidos cancelados: 53
Entregas atrasadas: 44


In [0]:
print("Mart Comercial - maiores receitas")
display(
    mart_performance_comercial
    .orderBy(F.desc("receita_liquida"))
    .limit(10)
)

print("Mart Produtos - maiores receitas por item")
display(
    mart_performance_produtos
    .orderBy(F.desc("receita_item"))
    .limit(10)
)

print("Mart Entregas - maiores volumes de atraso")
display(
    mart_operacional_entregas
    .orderBy(F.desc("qtd_entregas_atrasadas"))
    .limit(10)
)

print("Mart Atendimento - maiores volumes de ocorrência")
display(
    mart_atendimento_ocorrencias
    .orderBy(F.desc("qtd_ocorrencias"))
    .limit(10)
)

Mart Comercial - maiores receitas


data_pedido,ano_mes,estado_cliente,segmento_cliente,porte_cliente,seller_id,seller_name,channel_id,nome_canal,tipo_canal,regional_code,regional_name,qtd_pedidos,receita_bruta,valor_desconto,receita_liquida,qtd_pedidos_cancelados,ticket_medio,taxa_cancelamento
2025-09-16,2025-09,MG,Indústria,PEQUENA,V029,Vendedor 29,CH02,Parceiros,Indireto,S,Sul,1,17266.5,0.0,17266.5,1,8633.25,1.0
2025-09-13,2025-09,null,null,null,V032,Vendedor 32,CH04,Field Sales,Direto,N,Norte,1,14933.87,0.0,14933.87,0,14933.87,0.0
2025-05-07,2025-05,MG,Indústria,PEQUENA,V014,Vendedor 14,CH01,Inside Sales,Direto,S,Sul,1,14918.58,0.0,14918.58,0,14918.58,0.0
2025-11-15,2025-11,MG,Varejo,PEQUENA,V038,Vendedor 38,CH05,E-commerce,digital,SUL,Região Sul,1,14885.3,0.0,14885.3,0,14885.3,0.0
2025-10-07,2025-10,PR,Educação,GRANDE,V003,Vendedor 3,CH03,Marketplace,Digital,SUL,Região Sul,1,14847.66,0.0,14847.66,0,14847.66,0.0
2025-04-30,2025-04,SP,Indústria,GRANDE,V013,Vendedor 13,CH02,Parceiros,Indireto,S,Sul,1,14726.97,0.0,14726.97,0,14726.97,0.0
2025-10-03,2025-10,RJ,Indústria,null,V004,Vendedor 4,CH02,Parceiros,Indireto,NE,Nordeste,1,14759.08,79.46,14684.46,0,14684.46,0.0
2025-12-26,2025-12,RJ,Educação,null,V010,Vendedor 10,CH02,Parceiros,Indireto,SE,Sudeste,1,14645.74,0.0,14645.74,0,14645.74,0.0
2025-05-27,2025-05,SC,Saúde,null,V016,Vendedor 16,CH05,E-commerce,digital,S,Sul,1,14600.77,0.0,14600.77,0,14600.77,0.0
2025-06-23,2025-06,SC,Indústria,MÉDIA,V023,Vendedor 23,CH05,E-commerce,digital,CO,Centro-Oeste,1,14547.81,0.0,14547.81,0,14547.81,0.0


Mart Produtos - maiores receitas por item


data_pedido,ano_mes,product_id,product_name,category,subcategory,product_family,qtd_pedidos,qtd_itens_vendidos,receita_item,preco_medio_unitario
2025-09-16,2025-09,P0055,Produto 55,Assinatura,null,Legacy,1,20.0,50766.6,2538.33
2025-06-21,2025-06,P0036,Produto 36,Serviços,Suporte,Legacy,1,12.0,29904.06,1805.51
2026-02-07,2026-02,P0026,Produto 26,Hardware,Sensor,Legacy,1,10.0,29742.3,2974.23
2026-01-30,2026-01,P0018,Produto 18,Assinatura,Mensal,Premium,1,10.0,29661.1,2966.11
2025-02-13,2025-02,P0056,Produto 56,Software,ETL,Legacy,1,10.0,29483.0,2948.3
2025-08-05,2025-08,P0035,Produto 35,Serviços,Suporte,Core,1,10.0,29442.2,2944.22
2025-04-30,2025-04,P0055,Produto 55,Assinatura,null,Legacy,1,10.0,29164.4,2916.44
2025-08-26,2025-08,P0031,Produto 31,Assinatura,Mensal,Legacy,1,10.0,29145.6,2914.56
2025-12-19,2025-12,P0025,Produto 25,Software,CRM,Legacy,1,10.0,28851.8,2885.18
2025-07-31,2025-07,P0041,Produto 41,Hardware,Coletor,Core,1,10.0,28409.5,2840.95


Mart Entregas - maiores volumes de atraso


data_envio,ano_mes,destination_state,destination_city,carrier_name,delivery_mode,delivery_status,segmento_cliente,porte_cliente,qtd_entregas,qtd_pedidos_com_entrega,qtd_entregues,qtd_entregas_atrasadas,prazo_medio_entrega_dias,custo_total_entrega,custo_medio_entrega,taxa_atraso_entrega
2025-11-28,2025-11,SC,Florianópolis,EntregaJá,rodoviario,ATRASADO,Saúde,null,1,1,1,1,2.0,26.71,26.71,1.0
2025-08-18,2025-08,PR,Londrina,EntregaJá,rodoviario,ATRASADO,Serviços,null,1,1,1,1,1.0,26.25,26.25,1.0
2025-10-16,2025-10,SP,Ribeirão Preto,null,Aéreo,ATRASADO,Varejo,GRANDE,1,1,1,1,1.0,26.66,26.66,1.0
2025-04-28,2025-04,RJ,Rio de Janeiro,EntregaJá,Aéreo,ATRASADO,Varejo,GRANDE,1,1,1,1,11.0,169.74,169.74,1.0
2025-03-31,2025-03,SP,Campinas,EntregaJá,rodoviario,ATRASADO,null,null,1,1,1,1,12.0,204.49,204.49,1.0
2025-01-21,2025-01,RJ,Rio de Janeiro,null,Aéreo,ATRASADO,Indústria,GRANDE,1,1,1,1,6.0,147.11,147.11,1.0
2025-03-09,2025-03,RJ,Volta Redonda,LogFast,Aéreo,ATRASADO,Saúde,MÉDIA,1,1,1,1,5.0,325.24,325.24,1.0
2025-08-12,2025-08,MG,Contagem,LogFast,Rodoviário,ATRASADO,null,GRANDE,1,1,1,1,6.0,300.27,300.27,1.0
2025-11-22,2025-11,SC,Florianópolis,null,Aéreo,ATRASADO,Varejo,GRANDE,1,1,1,1,3.0,null,null,1.0
2025-04-15,2025-04,MG,Contagem,LogFast,Rodoviário,ATRASADO,null,null,1,1,1,1,10.0,219.75,219.75,1.0


Mart Atendimento - maiores volumes de ocorrência


data_ocorrencia,ano_mes,event_type,severity,status_ocorrencia,estado_cliente,segmento_cliente,porte_cliente,qtd_ocorrencias,qtd_pedidos_com_ocorrencia,qtd_ocorrencias_abertas,taxa_ocorrencia_aberta
2025-02-26,2025-02,cancel_request,MEDIUM,null,SP,Saúde,PEQUENA,1,1,0,0.0
2025-11-27,2025-11,refund,LOW,null,SC,Educação,MÉDIA,1,1,0,0.0
2025-05-15,2025-05,null,MEDIUM,FECHADO,RJ,Varejo,GRANDE,1,1,0,0.0
2026-01-22,2026-01,delay,HIGH,FECHADO,RJ,null,GRANDE,1,1,0,0.0
2026-01-03,2026-01,cancel_request,MEDIUM,ABERTO,SC,Serviços,GRANDE,1,1,1,1.0
2025-10-14,2025-10,Delay,MEDIUM,ABERTO,PR,null,PEQUENA,1,1,1,1.0
2026-02-07,2026-02,refund,null,null,SP,null,GRANDE,1,1,0,0.0
2025-12-03,2025-12,delay,HIGH,ABERTO,MG,Saúde,null,1,1,1,1.0
2025-08-24,2025-08,complaint,null,ABERTO,SP,Educação,GRANDE,1,1,1,1.0
2025-04-04,2025-04,cancel_request,MEDIUM,ABERTO,null,null,null,1,1,1,1.0
